In [1]:
import os
import datetime
import logging
import pandas as pd
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, LongType

from helpers_spark.serialization import deserialize_json_columns, deserialize_json_columns


# from ingest_helpers.spark_processing_helpers2 import build_tree, normalize_tree, sort_tree, select_tree,  explode_nested, split_nested

# from ingest_helpers.spark_processing_helpers2_test import apply_enrich

# -----------------------------
# Logging setup
# -----------------------------
LOG_DIR = Path(os.getenv("LOG_DIR", "/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "discogs_silver_ingest.log"

logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(LOG_FILE)],
    )

logger.info(f"Logging to {LOG_FILE}")

# -----------------------------
# Environment / config
# -----------------------------
DATA_DIR = os.getenv("DATA_DIR")
BRONZE_DATA_DIR = os.path.join(DATA_DIR, "bronze","artists")

# -----------------------------
# Spark session
# -----------------------------
spark = (
    SparkSession.builder.appName("discogs-silver-ingest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")



logger.info(f"loading {BRONZE_DATA_DIR}")        
df = spark.read.format("delta").load(BRONZE_DATA_DIR)

2026-02-25 12:15:51,512 | INFO     | __main__ | Logging to /logs/discogs_silver_ingest.log
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 12:15:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/25 12:15:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2026-02-25 12:15:54,880 | INFO     | __main__ | loading /data_tests/bronze/artists


In [2]:
df.columns

['aliases',
 'data_quality',
 'groups',
 'id',
 'members',
 'name',
 'namevariations',
 'profile',
 'realname',
 'urls',
 'root_hash',
 'last_dump_update',
 'groups_hash']

In [3]:
df.select("id", "name","groups").show()

+-----+----------------+--------------------+
|   id|            name|              groups|
+-----+----------------+--------------------+
|66474|        Hysterie|                NULL|
| 5797|             ATB|                NULL|
|13961|           Sting|                NULL|
|12971|         Exuviae|                NULL|
|11498|   Psi Performer|                NULL|
| 6400|          Asylum|                NULL|
|80159|       Zoe Ellis|{"name":[{"@id":8...|
|34226|       DJ Design|                NULL|
|58078|        Lena (2)|                NULL|
|22975|    Ink & J. Dub|                NULL|
|51878|    Anenzephalia|                NULL|
|51166|    Helena Brown|                NULL|
|13473|     Belle Ville|                NULL|
| 8040|            Kava|                NULL|
|29197|  Kazushi Matsui|                NULL|
|32181|Pierre Schaeffer|{"name":[{"@id":1...|
|40527|       Tahiti 80|                NULL|
| 5039| Twisted Science|                NULL|
| 7570| Peace Orchestra|          

In [18]:

groups_schema = StructType([
    StructField("name", ArrayType(
        StructType([
            StructField("@id", LongType()),
            StructField("_text", StringType())
        ])
    ))
])

df_groups = (
    df
    .withColumn("groups_struct", F.from_json("groups", groups_schema))
    .withColumn("group", F.explode_outer("groups_struct.name"))
    .select(
        F.col("id").alias("artist_id"),
        F.col("name").alias("artist_name"),
        F.col("group.@id").alias("group_id"),
        F.col("group._text").alias("group_name"),
        "last_dump_update",
            "groups_hash"
    ).filter(F.col("group_id").isNotNull())
)

df_groups.show()

+---------+--------------+--------+--------------------+----------------+--------------------+
|artist_id|   artist_name|group_id|          group_name|last_dump_update|         groups_hash|
+---------+--------------+--------+--------------------+----------------+--------------------+
|     2885|        Leekon|    2883|                Swap|        20251101|5638dc819d862755c...|
|    54208|      Mei Lwun|    3333|           Soulstice|        20251101|0a1775fecc7343e66...|
|    35625|    Dana Vlcek|    8604|      Abstract Truth|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   16804|                Konk|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   54820|               MKS-7|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   79333|      Lounge Lizards|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek| 9065191|        The Sync (3)|        20251101|724fd0c2d1a3f9fd5...|
|     1233|John Acquaviva|    1699|          Cyber

In [19]:
    groups_schema = StructType([
        StructField("name", ArrayType(
            StructType([
                StructField("@id", LongType()),
                StructField("_text", StringType())
            ])
        ))
    ])

    df_artists_groups_complete = (
        df
        .withColumn("groups_struct", F.from_json("groups", groups_schema))
        .withColumn("group", F.explode_outer("groups_struct.name"))
        .select(
            F.col("id").alias("artist_id"),
            F.col("name").alias("artist_name"),
            F.col("group.@id").alias("group_id"),
            F.col("group._text").alias("group_name"),
            "last_dump_update",
            "groups_hash"
        )
        .filter(F.col("group_id").isNotNull())
    )

df_artists_groups_complete.show()

+---------+--------------+--------+--------------------+----------------+--------------------+
|artist_id|   artist_name|group_id|          group_name|last_dump_update|         groups_hash|
+---------+--------------+--------+--------------------+----------------+--------------------+
|     2885|        Leekon|    2883|                Swap|        20251101|5638dc819d862755c...|
|    54208|      Mei Lwun|    3333|           Soulstice|        20251101|0a1775fecc7343e66...|
|    35625|    Dana Vlcek|    8604|      Abstract Truth|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   16804|                Konk|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   54820|               MKS-7|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek|   79333|      Lounge Lizards|        20251101|724fd0c2d1a3f9fd5...|
|    35625|    Dana Vlcek| 9065191|        The Sync (3)|        20251101|724fd0c2d1a3f9fd5...|
|     1233|John Acquaviva|    1699|          Cyber

In [22]:
df_artists_groups_complete.schema

StructType([StructField('artist_id', LongType(), True), StructField('artist_name', StringType(), True), StructField('group_id', LongType(), True), StructField('group_name', StringType(), True), StructField('last_dump_update', StringType(), True), StructField('groups_hash', StringType(), True)])

In [20]:
    groups_schema = StructType([
        StructField("name", ArrayType(
            StructType([
                StructField("@id", LongType()),
                StructField("_text", StringType())
            ])
        ))
    ])

    df_artists_groups = (
        df
        .withColumn("groups_struct", F.from_json("groups", groups_schema))
        .withColumn("group", F.explode_outer("groups_struct.name"))
        .select(
            F.col("id").alias("artist_id"),
            F.col("group.@id").alias("group_id"),
            "last_dump_update",
            "groups_hash"
        )
        .filter(F.col("group_id").isNotNull())
    )

df_artists_groups.sort(["artist_id","group_id"]).show()

+---------+--------+----------------+--------------------+
|artist_id|group_id|last_dump_update|         groups_hash|
+---------+--------+----------------+--------------------+
|        3|   34803|        20251101|701b9af5034b20405...|
|        3|   55692|        20251101|701b9af5034b20405...|
|        3|   55693|        20251101|701b9af5034b20405...|
|        3|  579249|        20251101|701b9af5034b20405...|
|        3|  844878|        20251101|701b9af5034b20405...|
|        4|   10281|        20251101|873bbf681828de89a...|
|        4|   30969|        20251101|873bbf681828de89a...|
|        4|   35683|        20251101|873bbf681828de89a...|
|        4|   60367|        20251101|873bbf681828de89a...|
|        4|  239403|        20251101|873bbf681828de89a...|
|        5|    1320|        20251101|e72404d3f0e1fde74...|
|        5|    6874|        20251101|e72404d3f0e1fde74...|
|        5|   10281|        20251101|e72404d3f0e1fde74...|
|        5|   20864|        20251101|e72404d3f0e1fde74..

In [21]:
    groups_schema = StructType([
        StructField("name", ArrayType(
            StructType([
                StructField("@id", LongType()),
                StructField("_text", StringType())
            ])
        ))
    ])

    df_groups = (
        df
        .withColumn("groups_struct", F.from_json("groups", groups_schema))
        .withColumn("group", F.explode_outer("groups_struct.name"))
        .select(
            F.col("group.@id").alias("group_id"),
            F.col("group._text").alias("group_name"),
            "last_dump_update",
            "groups_hash"
        )
        .filter(F.col("group_id").isNotNull())
        .distinct()
    )

df_groups.sort("group_id").show()


+--------+--------------------+----------------+--------------------+
|group_id|          group_name|last_dump_update|         groups_hash|
+--------+--------------------+----------------+--------------------+
|       2|Mr. James Barth &...|        20251101|e40b7f8159d17a893...|
|       2|Mr. James Barth &...|        20251101|3a37de883132013c5...|
|       8|       Mood II Swing|        20251101|7f5b41e09a31350a0...|
|       8|       Mood II Swing|        20251101|f04dd13d6ae965286...|
|      13|               Blaze|        20251101|5806680193eecfc79...|
|      13|               Blaze|        20251101|85b927f4d76624016...|
|      13|               Blaze|        20251101|696e90771f5617862...|
|      19|    Sound Associates|        20251101|05f06a3c80ff4ceca...|
|      19|    Sound Associates|        20251101|1cc72dd1bb2157869...|
|      22|            DATacide|        20251101|f88ccb9e53efdff6c...|
|      30|   Groove Collective|        20251101|8978cd1fc90e35fc6...|
|      30|   Groove 

In [12]:
    df_artists_table = (
        df
        .select(
            F.col("id").alias("artist_id"),
            F.col("name").alias("artist_name")
        )
    )

df_artists_table.sort("artist_id").show()

+---------+--------------------+
|artist_id|         artist_name|
+---------+--------------------+
|        1|       The Persuader|
|        2|Mr. James Barth &...|
|        3|           Josh Wink|
|        4|       Johannes Heil|
|        5|          Heiko Laux|
|        6|              K.A.B.|
|        7|            Sylk 130|
|        8|       Mood II Swing|
|        9|        Care Company|
|       11|            DJ Dozia|
|       13|               Blaze|
|       14|    Eight Miles High|
|       16|     Christian Smith|
|       17|         John Selway|
|       19|    Sound Associates|
|       20|             Percy X|
|       21|         Faze Action|
|       22|            DATacide|
|       23|          Alex Hi-Fi|
|       24|          Atom Heart|
+---------+--------------------+
only showing top 20 rows
